In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [3]:
# Carga de datos
# Modificar la ruta a la ubicacion del dataset
df = pd.read_excel("C:\\Users\\USER\\Downloads\\base-de-datos-violencia-intrafamiliar-ano-2024_v3.xlsx")
df.head()

,HEC_DIA,HEC_MES,HEC_ANO,HEC_DEPTO,HEC_DEPTOMCPIO,HEC_TIPAGRE,NUMERO_BOLETA,DIA_EMISION,MES_EMISION,ANO_EMISION,...,ARTICULOCODPEN2,ARTICULOCODPEN3,ARTICULOCODPEN4,ARTICULOTRAS1,ARTICULOTRAS2,ARTICULOTRAS3,ARTICULOTRAS4,MEDIDAS_SEGURIDAD,TIPO_MEDIDA,ORGANISMO_REMITE
0,4,11,2024,1,110,1122,367,4,11,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,24,3,2024,2,202,1222,5,25,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,IJ,18.0
2,99,99,9999,1,101,1122,430,2,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28,3,2024,2,202,1122,6,28,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,AIJ,18.0
4,12,7,2024,7,706,2122,16,24,7,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,IJ,18.0


In [4]:
print("Original rows:", df.shape[0])

# Violencia fisica =1 , no fisica = 2
physical_aggression_codes = [1222, 2212, 1122,1212,1221,2211,1112,1121,1211,1111]
df["TARGET_PHYSICAL"] = df["HEC_TIPAGRE"].apply(
    lambda x: 1 if x in physical_aggression_codes else 0
)
df.head()


Original rows: 36609


,HEC_DIA,HEC_MES,HEC_ANO,HEC_DEPTO,HEC_DEPTOMCPIO,HEC_TIPAGRE,NUMERO_BOLETA,DIA_EMISION,MES_EMISION,ANO_EMISION,...,ARTICULOCODPEN3,ARTICULOCODPEN4,ARTICULOTRAS1,ARTICULOTRAS2,ARTICULOTRAS3,ARTICULOTRAS4,MEDIDAS_SEGURIDAD,TIPO_MEDIDA,ORGANISMO_REMITE,TARGET_PHYSICAL
0,4,11,2024,1,110,1122,367,4,11,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,24,3,2024,2,202,1222,5,25,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,IJ,18.0,1
2,99,99,9999,1,101,1122,430,2,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,28,3,2024,2,202,1122,6,28,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,AIJ,18.0,1
4,12,7,2024,7,706,2122,16,24,7,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,IJ,18.0,0


In [14]:

predictors = df[[
    "VIC_SEXO", "VIC_EDAD", "VIC_ALFAB", "VIC_ESCOLARIDAD",
    "VIC_EST_CIV", "VIC_TRABAJA", "VIC_OCUP",
    "AGR_SEXO", "AGR_EDAD", "AGR_TRABAJA", "AGR_OCUP",
    "VIC_REL_AGR",
    "OTRAS_VICTIMAS"
]].dropna()

print("Rows:", predictors.shape[0])



Rows: 10803


In [15]:
model_data = predictors.copy()
model_data["TARGET_PHYSICAL"] = df.loc[predictors.index, "TARGET_PHYSICAL"]

X = model_data.drop(columns=["TARGET_PHYSICAL"])
y = model_data["TARGET_PHYSICAL"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=123, stratify=y
)


In [16]:
input_dim = X_train.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=60,
    batch_size=64,
    validation_data=(X_test, y_test)
)

loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nLoss: {loss:.4f}, Accuracy: {accuracy:.4f}")


Epoch 1/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.5454 - loss: 139.3351 - val_accuracy: 0.4054 - val_loss: 41.6955
Epoch 2/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5475 - loss: 51.3803 - val_accuracy: 0.4030 - val_loss: 24.3460
Epoch 3/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5320 - loss: 25.6190 - val_accuracy: 0.6316 - val_loss: 5.8470
Epoch 4/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5249 - loss: 10.9454 - val_accuracy: 0.5035 - val_loss: 1.4328
Epoch 5/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5386 - loss: 7.3425 - val_accuracy: 0.6316 - val_loss: 3.3891
Epoch 6/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5276 - loss: 5.2498 - val_accuracy: 0.6057 - val_loss: 0.8272
Epoch 7/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5459 - loss: 4.9674 - val_accuracy: 0.6316 - val_loss: 2.8085
Epoch 8/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5339 - loss: 3.2437 - val_accu

In [18]:

# prediccion individual
new_case = pd.DataFrame([{
    "VIC_SEXO": 2, # mujer
    "VIC_EDAD": 28,
    "VIC_ALFAB": 1, # alfabeta
    "VIC_ESCOLARIDAD": 59, # universitario
    "VIC_EST_CIV": 2, # casada
    "VIC_TRABAJA": 2, # No trabaja por un salario o ingreso
    "VIC_OCUP": 2631, # economistas
    "AGR_SEXO": 1, # Hombre
    "AGR_EDAD": 30,
    "AGR_TRABAJA": 1, # si trabaja por un salario o ingreso
    "AGR_OCUP": 2141,
    "VIC_REL_AGR": 1, # esposo
    "OTRAS_VICTIMAS": 0
}])

new_case = new_case[X_train.columns]

prob = model.predict(new_case)[0][0]
prediction = 1 if prob >= 0.5 else 0

print("\nProbabilidad prevista de violencia FÍSICA:", prob)
print("Clase prevista (0=no física, 1=física):", prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step

Probabilidad prevista de violencia FÍSICA: 0.43097353
Clase prevista (0=no física, 1=física): 0
